### 1. Setup
<hr>

In [ ]:
# --- Imports ---
import sys
import time
import numpy as np
import torch
from torch.utils.data import Dataset
import platform
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments
)

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
# --- Experiment Configuration ---
MODEL_NAME = "nghuyong/ernie-3.0-base-zh"

# Model / input
NUM_LABELS = 2
MAX_LENGTH = 64

# Inference benchmarks process one sample per call (batch size = 1)
BATCH_SIZE = 1

# Benchmark
WARMUP_RUNS = 10
NUM_RUNS = 100

# CPU thread tuning
THREAD_COUNTS = (1, 2, 4) # AWS c7g.xlarge: 4 vCPUs
INTER_THREADS = 1

# Fine-tuning
RUN_FINETUNING = False

In [ ]:
# --- Device setup ---
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print(f"Device: {device}")
print(platform.processor())
print(platform.machine())

### 2. Load Dataset
<hr>

In [ ]:
from src.utils.load_dataset import load_dataset

train_df = load_dataset("train")
dev_df = load_dataset("dev")

# The ChnSentiCorp test split does not contain labels.
# Use the development split for validation and quality evaluation.

### 3. Load Pretrained Ernie
<hr>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model = model.to(device)

### 4. Fine-tune ERNIE
<hr>

In [ ]:
class ChnSentiCorpDataset(Dataset):

    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text_a"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": self.labels[idx],
        }

In [ ]:
train_dataset = ChnSentiCorpDataset(train_df, tokenizer)
dev_dataset = ChnSentiCorpDataset(dev_df, tokenizer)

In [ ]:
if RUN_FINETUNING:

    from src.evaluation.metrics import compute_metrics

    training_args = TrainingArguments(
        output_dir="../models/checkpoints",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        gradient_accumulation_steps=8,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # save the fine-tuned PyTorch models
    trainer.save_model("../models/ernie-finetuned")
    tokenizer.save_pretrained("../models/ernie-finetuned")

### 5. PyTorch FP32 Baseline
<hr>

In [ ]:
from src.benchmark.benchmark_pytorch import benchmark_pytorch

FINETUNED_MODEL_PATH = "../models/ernie-finetuned"

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_PATH)

# Load fine-tuned model
model = AutoModelForSequenceClassification.from_pretrained(
    FINETUNED_MODEL_PATH
)

# Select benchmark sample
text = dev_df["text_a"].iloc[0]

# Benchmark PyTorch inference
pytorch_results = benchmark_pytorch(
    model=model,
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    benchmark_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)

In [ ]:
# from src.profiling.profile_pytorch import profile_pytorch

# Optional profiling
# profile_pytorch(
#     model=model,
#     tokenizer=tokenizer,
#     text=text,
#     warmup_runs=WARMUP_RUNS,
#     max_length=MAX_LENGTH,
#     row_limit=20,
# )

### 6. Export to ONNX
<hr>

In [ ]:
from src.optimization.export_onnx import export_onnx, validate_onnx

onnx_model_path = "../models/ernie_finetuned.onnx"

export_onnx(
    model=model,
    tokenizer=tokenizer,
    text=text,
    output_path=onnx_model_path,
    max_length=MAX_LENGTH,
    opset_version=17,
)

validate_onnx(onnx_model_path)

### 7. ONNX FP32 Baseline
<hr>

In [ ]:
from src.benchmark.benchmark_onnx import benchmark_onnx

onnx_result = benchmark_onnx(
    model_path=onnx_model_path,
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    benchmark_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)

### 8. ONNX Graph Optimization
<hr>

In [ ]:
from src.optimization.optimize_onnx import benchmark_optimization_levels

optimization_results, best_optimization_level = benchmark_optimization_levels(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    benchmark_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)

print(best_optimization_level)

### 9. CPU Thread Tuning
<hr>

In [ ]:
from src.optimization.tune_threads import tune_threads

thread_results, best_thread_config = tune_threads(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    text=text,
    thread_counts=THREAD_COUNTS,     
    inter_threads=INTER_THREADS,
    warmup_runs=WARMUP_RUNS,
    benchmark_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
    optimization_level=best_optimization_level,
)

In [ ]:
print("=" * 75)
print("Thread Tuning Summary")
print("=" * 75)

print(
    f"{'Intra':<10}"
    f"{'Avg (ms)':<12}"
    f"{'Median':<12}"
    f"{'P95':<12}"
    f"{'Throughput':<15}"
)

for result in thread_results:
    print(
        f"{result['intra_threads']:<10}"
        f"{result['average']:<12.2f}"
        f"{result['median']:<12.2f}"
        f"{result['p95']:<12.2f}"
        f"{result['throughput']:<15.2f}"
    )

In [ ]:
print()
print("Best configuration")
print("-" * 40)

print(f"Intra-op threads : {best_thread_config['intra_threads']}")
print(f"Inter-op threads : {best_thread_config['inter_threads']}")
print(f"Median latency   : {best_thread_config['median']:.2f} ms")
print(f"Average latency  : {best_thread_config['average']:.2f} ms")
print(f"Throughput       : {best_thread_config['throughput']:.2f} samples/sec")

### 10. Dynamic INT8 Quantization
<hr>

In [ ]:
from src.optimization.quantize import quantize_model

quantize_model(
    input_model_path="../models/ernie_finetuned.onnx",
    relaxed_model_path="../models/ernie_finetuned_relaxed.onnx",
    output_model_path="../models/ernie_finetuned_int8.onnx",
)

In [ ]:
from pathlib import Path

# Helper -> compute model size
def model_size_mb(*paths):
    total_bytes = sum(
        Path(path).stat().st_size
        for path in paths
        if Path(path).exists()
    )

    return total_bytes / (1024 ** 2)

### 11. Model Size Comparison
<hr>

In [ ]:
fp32_size = model_size_mb(
    "../models/ernie_finetuned.onnx",
    "../models/ernie_finetuned.onnx.data",
)

int8_size = model_size_mb(
    "../models/ernie_finetuned_int8.onnx",
    "../models/ernie_finetuned_int8.onnx.data",
)

print(f"FP32 size : {fp32_size:.2f} MB")
print(f"INT8 size : {int8_size:.2f} MB")
print(f"Reduction : {(1 - int8_size / fp32_size) * 100:.2f}%")

### 12. Model Quality Evaluation
<hr>

In [ ]:
from src.evaluation.evaluate_onnx import evaluate_onnx

y_true = dev_df["label"].to_numpy()

int8_metrics = evaluate_onnx(
    model_path="../models/ernie_finetuned_int8.onnx",
    tokenizer=tokenizer,
    texts=dev_df["text_a"],
    labels=y_true,
    max_length=MAX_LENGTH,
)

print(int8_metrics)

In [ ]:
fp32_metrics = evaluate_onnx(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    texts=dev_df["text_a"],
    labels=y_true,
    max_length=MAX_LENGTH,
)

print(fp32_metrics)

In [ ]:
print(f"Accuracy change:  {(int8_metrics['accuracy'] - fp32_metrics['accuracy']) * 100:+.2f}%")
print(f"Precision change: {(int8_metrics['precision'] - fp32_metrics['precision']) * 100:+.2f}%")
print(f"Recall change:    {(int8_metrics['recall'] - fp32_metrics['recall']) * 100:+.2f}%")
print(f"F1 change:        {(int8_metrics['f1'] - fp32_metrics['f1']) * 100:+.2f}%")

### 13. INT8 Graph Optimization
<hr>

In [ ]:
int8_optimization_results, best_int8_optimization_level = (
    benchmark_optimization_levels(
        model_path="../models/ernie_finetuned_int8.onnx",
        tokenizer=tokenizer,
        text=text,
        warmup_runs=WARMUP_RUNS,
        benchmark_runs=NUM_RUNS,
        max_length=MAX_LENGTH,
    )
)

### 14. INT8 Thread Tuning
<hr>

In [ ]:
int8_thread_results, best_int8_thread_config = tune_threads(
    model_path="../models/ernie_finetuned_int8.onnx",
    tokenizer=tokenizer,
    text=text,
    thread_counts=THREAD_COUNTS,
    inter_threads=INTER_THREADS,
    warmup_runs=WARMUP_RUNS,
    benchmark_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
    optimization_level=best_int8_optimization_level,
)

In [ ]:
print("=" * 75)
print("Thread Tuning Summary")
print("=" * 75)

print(
    f"{'Intra':<10}"
    f"{'Avg (ms)':<12}"
    f"{'Median':<12}"
    f"{'P95':<12}"
    f"{'Throughput':<15}"
)

for result in int8_thread_results:
    print(
        f"{result['intra_threads']:<10}"
        f"{result['average']:<12.2f}"
        f"{result['median']:<12.2f}"
        f"{result['p95']:<12.2f}"
        f"{result['throughput']:<15.2f}"
    )

In [ ]:
print()
print("Best configuration")
print("-" * 40)

print(f"Intra-op threads : {best_int8_thread_config['intra_threads']}")
print(f"Inter-op threads : {best_int8_thread_config['inter_threads']}")
print(f"Median latency   : {best_int8_thread_config['median']:.2f} ms")
print(f"Average latency  : {best_int8_thread_config['average']:.2f} ms")
print(f"Throughput       : {best_int8_thread_config['throughput']:.2f} samples/sec")

### 15. Final Results
<hr>

In [ ]:
print("=" * 55)
print("Model Quality Comparison")
print("=" * 55)

print(
    f"{'Model':<15}"
    f"{'Accuracy':<15}"
    f"{'F1':<15}"
)

print("-" * 55)

print(
    f"{'ONNX FP32':<15}"
    f"{fp32_metrics['accuracy'] * 100:<15.2f}"
    f"{fp32_metrics['f1'] * 100:<15.2f}"
)

print(
    f"{'ONNX INT8':<15}"
    f"{int8_metrics['accuracy'] * 100:<15.2f}"
    f"{int8_metrics['f1'] * 100:<15.2f}"
)

print("=" * 55)

print(f"Accuracy change : {(int8_metrics['accuracy'] - fp32_metrics['accuracy']) * 100:+.2f} pp")
print(f"F1 change       : {(int8_metrics['f1'] - fp32_metrics['f1']) * 100:+.2f} pp")